# CMPT 310 – Pathfinding Project
**Group: Pathfinders**

This notebook covers **map / problem generation** and **visualization**.
Search algorithms (BFS, DFS, Greedy, etc.) will consume the grid produced here.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import random
import math

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

: 

In [ ]:
# ── Terrain constants ────────────────────────────────────────────────────────

# Movement cost to ENTER each terrain type (used by search algorithms).
# Obstacles are impassable; their cost is represented as infinity.
TERRAIN_COSTS = {
    "plain":    1,
    "forest":   3,
    "mountain": 5,
    "swamp":    4,
    "obstacle": math.inf,
    "target":   1,   # same cost as plain when a search algorithm steps onto it
}

# Whether each terrain type allows movement through it.
TERRAIN_PASSABLE = {
    "plain":    True,
    "forest":   True,
    "mountain": True,
    "swamp":    True,
    "obstacle": False,
    "target":   True,
}

# Display colors for each terrain type (used by the visualizer).
TERRAIN_COLORS = {
    "plain":    "#F5F0C8",   # light yellow
    "forest":   "#4CAF50",   # medium green
    "mountain": "#8D7B5E",   # warm grey-brown
    "swamp":    "#4DB6AC",   # teal
    "obstacle": "#37474F",   # dark slate
    "target":   "#FFF176",   # bright yellow (star drawn on top)
}

# Short label shown inside each cell alongside the cost number.
TERRAIN_LABELS = {
    "plain":    "",
    "forest":   "F",
    "mountain": "M",
    "swamp":    "S",
    "obstacle": "X",
    "target":   "★",
}

## Map / Problem Generation

`generate_map` creates a reproducible 2-D grid with:
- randomised terrain types and movement costs
- a guaranteed-passable target cell placed at a random position
- configurable obstacle density and terrain frequency

In [ ]:
def generate_map(
    rows: int = 10,
    cols: int = 10,
    seed: int = None,
    obstacle_ratio: float = 0.15,
    terrain_weights: dict = None,
) -> dict:
    """
    Generate a 2-D terrain grid for pathfinding experiments.

    Parameters
    ----------
    rows : int
        Number of rows in the grid.
    cols : int
        Number of columns in the grid.
    seed : int or None
        Random seed for reproducibility.  None = random seed chosen automatically.
    obstacle_ratio : float
        Fraction of cells that become impassable obstacles (0.0 – 0.5 recommended).
    terrain_weights : dict or None
        Probability weight for each passable terrain type.
        Keys: "plain", "forest", "mountain", "swamp".
        Missing keys default to the built-in weights.
        Values are normalised automatically so they do not need to sum to 1.

    Returns
    -------
    dict with keys:
        "grid"   – 2-D list (rows × cols) of cell dicts.
                   Each cell dict: {"type": str, "cost": float, "passable": bool}
        "rows"   – int
        "cols"   – int
        "target" – (row, col) tuple of the target cell
        "seed"   – int seed that was actually used
    """
    # ── Resolve seed ─────────────────────────────────────────────────────────
    if seed is None:
        seed = random.randint(0, 10_000)
    rng = random.Random(seed)

    # ── Resolve terrain weights ───────────────────────────────────────────────
    default_weights = {
        "plain":    0.55,
        "forest":   0.20,
        "mountain": 0.13,
        "swamp":    0.12,
    }
    if terrain_weights:
        for key in terrain_weights:
            if key not in default_weights:
                raise ValueError(f"Unknown terrain type in terrain_weights: '{key}'")
        merged = {**default_weights, **terrain_weights}
    else:
        merged = default_weights

    terrain_types = list(merged.keys())
    total = sum(merged.values())
    weights = [merged[t] / total for t in terrain_types]

    # ── Build grid ───────────────────────────────────────────────────────────
    # Step 1: assign passable terrain to every cell.
    grid = []
    for r in range(rows):
        row = []
        for c in range(cols):
            terrain = rng.choices(terrain_types, weights=weights, k=1)[0]
            row.append({
                "type":     terrain,
                "cost":     TERRAIN_COSTS[terrain],
                "passable": TERRAIN_PASSABLE[terrain],
            })
        grid.append(row)

    # Step 2: randomly convert some cells to obstacles.
    num_obstacles = int(rows * cols * obstacle_ratio)
    all_positions = [(r, c) for r in range(rows) for c in range(cols)]
    obstacle_positions = rng.sample(all_positions, num_obstacles)
    for (r, c) in obstacle_positions:
        grid[r][c] = {
            "type":     "obstacle",
            "cost":     TERRAIN_COSTS["obstacle"],
            "passable": False,
        }

    # Step 3: place the target on a random passable cell.
    passable_positions = [
        (r, c)
        for r in range(rows)
        for c in range(cols)
        if grid[r][c]["passable"]
    ]
    if not passable_positions:
        raise ValueError(
            "obstacle_ratio is too high – no passable cell left for the target."
        )
    target_pos = rng.choice(passable_positions)
    tr, tc = target_pos
    grid[tr][tc] = {
        "type":     "target",
        "cost":     TERRAIN_COSTS["target"],
        "passable": True,
    }

    return {
        "grid":   grid,
        "rows":   rows,
        "cols":   cols,
        "target": target_pos,
        "seed":   seed,
    }

## Visualization

`visualize_map` renders the grid with:
- colour-coded terrain types
- movement cost displayed inside each passable cell
- star marker on the target cell
- highlighted border edges indicating valid agent entry sides
- legend for terrain types

In [ ]:
def visualize_map(
    problem: dict,
    ax=None,
    show_cost: bool = True,
    show_entry_borders: bool = True,
    title: str = None,
    figsize_per_cell: float = 0.65,
) -> None:
    """
    Render a problem dict (returned by generate_map) as a colour-coded grid.

    Parameters
    ----------
    problem : dict
        Output of generate_map().
    ax : matplotlib.axes.Axes or None
        Axes to draw on.  A new figure is created when None.
    show_cost : bool
        If True, print movement cost inside each passable cell.
    show_entry_borders : bool
        If True, highlight the four outer edges to indicate possible entry sides.
    title : str or None
        Custom title.  Defaults to an auto-generated string.
    figsize_per_cell : float
        Controls how large each cell appears in the figure.
    """
    grid = problem["grid"]
    rows = problem["rows"]
    cols = problem["cols"]
    seed = problem["seed"]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(
            figsize=(cols * figsize_per_cell + 2, rows * figsize_per_cell + 1.5)
        )

    # ── Draw cell rectangles ─────────────────────────────────────────────────
    for r in range(rows):
        for c in range(cols):
            cell = grid[r][c]
            color = TERRAIN_COLORS[cell["type"]]

            # Row 0 is at the top; flip y so the grid reads naturally.
            y = rows - 1 - r

            rect = mpatches.FancyBboxPatch(
                (c + 0.02, y + 0.02),
                0.96, 0.96,
                boxstyle="round,pad=0.02",
                linewidth=0.5,
                edgecolor="#999999",
                facecolor=color,
            )
            ax.add_patch(rect)

            # Cost label inside each cell (target gets a star; obstacles get an X).
            if cell["type"] == "target":
                ax.text(
                    c + 0.5, y + 0.5, "★",
                    ha="center", va="center",
                    fontsize=14, color="#E53935", fontweight="bold",
                )
            elif not cell["passable"]:
                ax.text(
                    c + 0.5, y + 0.5, "✕",
                    ha="center", va="center",
                    fontsize=9, color="#ECEFF1",
                )
            elif show_cost:
                label = TERRAIN_LABELS[cell["type"]]
                cost_str = str(int(cell["cost"]))
                display = f"{label}\n{cost_str}" if label else cost_str
                ax.text(
                    c + 0.5, y + 0.5, display,
                    ha="center", va="center",
                    fontsize=7.5, color="#212121",
                    linespacing=1.4,
                )

    # ── Entry border highlights ──────────────────────────────────────────────
    # Thick blue lines along each outer edge show that agents may enter from
    # any of the four sides.  The actual entry-point selection is handled by
    # the search module.
    if show_entry_borders:
        border_color = "#1565C0"
        lw = 4
        ax.plot([0, cols], [rows, rows], color=border_color, linewidth=lw, solid_capstyle="round")
        ax.plot([0, cols], [0, 0],       color=border_color, linewidth=lw, solid_capstyle="round")
        ax.plot([0, 0],    [0, rows],    color=border_color, linewidth=lw, solid_capstyle="round")
        ax.plot([cols, cols], [0, rows], color=border_color, linewidth=lw, solid_capstyle="round")

        entry_style = dict(fontsize=7, color=border_color, fontweight="bold", alpha=0.85)
        ax.text(cols / 2, rows + 0.15, "▲ Entry", ha="center", va="bottom", **entry_style)
        ax.text(cols / 2, -0.15,       "▼ Entry", ha="center", va="top",    **entry_style)
        ax.text(-0.15,    rows / 2,    "◀ Entry", ha="right",  va="center", **entry_style)
        ax.text(cols + 0.15, rows / 2, "▶ Entry", ha="left",   va="center", **entry_style)

    # ── Axes cosmetics ───────────────────────────────────────────────────────
    ax.set_xlim(-0.6, cols + 0.6)
    ax.set_ylim(-0.6, rows + 0.6)
    ax.set_aspect("equal")

    # Column indices along the bottom, row indices along the left.
    ax.set_xticks(np.arange(cols) + 0.5)
    ax.set_xticklabels([str(c) for c in range(cols)], fontsize=7, color="#555")
    ax.set_yticks(np.arange(rows) + 0.5)
    ax.set_yticklabels([str(rows - 1 - r) for r in range(rows)], fontsize=7, color="#555")
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # ── Title ────────────────────────────────────────────────────────────────
    if title is None:
        target_r, target_c = problem["target"]
        title = (
            f"Terrain Map  ({rows}×{cols})   "
            f"seed={seed}   "
            f"target=({target_r},{target_c})"
        )
    ax.set_title(title, fontsize=11, pad=10)

    # ── Legend (only when we own the figure) ─────────────────────────────────
    legend_entries = [
        mpatches.Patch(facecolor=TERRAIN_COLORS[t], edgecolor="#999", label=lbl)
        for t, lbl in [
            ("plain",    "Plain  (cost 1)"),
            ("forest",   "Forest  (cost 3)"),
            ("mountain", "Mountain  (cost 5)"),
            ("swamp",    "Swamp  (cost 4)"),
            ("obstacle", "Obstacle  (impassable)"),
            ("target",   "Target  ★"),
        ]
    ]
    if standalone:
        fig.legend(
            handles=legend_entries,
            loc="lower center",
            ncol=3,
            fontsize=8,
            framealpha=0.9,
            bbox_to_anchor=(0.5, -0.02),
        )
        plt.tight_layout()
        plt.show()

## Demo – Default 10 × 10 Map

In [ ]:
# Generate and display a default 10×10 map.
problem = generate_map(rows=10, cols=10, seed=42)

print(f"Seed   : {problem['seed']}")
print(f"Size   : {problem['rows']} rows × {problem['cols']} cols")
print(f"Target : row={problem['target'][0]}, col={problem['target'][1]}")

visualize_map(problem)

## Parameter Exploration – Side-by-Side Comparison

Four maps with different seeds and configurations to show how parameters affect the terrain layout.

In [ ]:
configs = [
    # (subplot title, generate_map kwargs)
    (
        "Default 10×10  seed=1",
        dict(rows=10, cols=10, seed=1),
    ),
    (
        "Heavy forest 10×10  seed=2",
        dict(rows=10, cols=10, seed=2,
             terrain_weights={"forest": 0.55, "plain": 0.20, "mountain": 0.13, "swamp": 0.12}),
    ),
    (
        "Dense obstacles 12×12  seed=3",
        dict(rows=12, cols=12, seed=3, obstacle_ratio=0.30),
    ),
    (
        "Mountain terrain 8×8  seed=4",
        dict(rows=8, cols=8, seed=4,
             terrain_weights={"mountain": 0.50, "plain": 0.20, "forest": 0.15, "swamp": 0.15}),
    ),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (label, kwargs) in zip(axes, configs):
    prob = generate_map(**kwargs)
    visualize_map(prob, ax=ax, title=label)

# Shared legend below all four subplots.
legend_entries = [
    mpatches.Patch(facecolor=TERRAIN_COLORS[t], edgecolor="#999", label=lbl)
    for t, lbl in [
        ("plain",    "Plain  (cost 1)"),
        ("forest",   "Forest  (cost 3)"),
        ("mountain", "Mountain  (cost 5)"),
        ("swamp",    "Swamp  (cost 4)"),
        ("obstacle", "Obstacle  (impassable)"),
        ("target",   "Target  ★"),
    ]
]
fig.legend(
    handles=legend_entries,
    loc="lower center",
    ncol=6,
    fontsize=9,
    framealpha=0.9,
    bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle("Parameter Exploration – Side-by-Side Map Comparison", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()